In [1]:
"""
NLDC Daily PSP Report Downloader
==================================
Downloads all XLS files from the GRID-India API for the specified year ranges.

API endpoint : POST https://webapi.grid-india.in/api/v1/file
CDN base     : https://webcdn.grid-india.in/

Usage:
    python nldc_downloader.py

All XLS files will be saved to:
    ./nldc_xls/<year_range>/<filename>.xls

A download log (nldc_download_log.csv) is written so the script can
resume safely if interrupted — already-downloaded files are skipped.
"""

import os
import time
import json
import requests
import urllib3
import csv
from pathlib import Path
from datetime import datetime

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# ... all the configuration variables ...
# ... all the functions ...
# STOP before main() — do NOT include the main() call here

# ── Configuration ──────────────────────────────────────────────────────────────

API_URL      = "https://webapi.grid-india.in/api/v1/file"
CDN_BASE     = "https://webcdn.grid-india.in/"
OUTPUT_DIR   = Path("./nldc_xls")
LOG_FILE     = Path("./nldc_download_log.csv")

# Year ranges to download
YEAR_RANGES  = ["2025-26", "2024-25", "2023-24", "2022-23"]

# XLS mime types (exclude PDF)
XLS_MIMES    = {
    "application/vnd.ms-excel",
    "attachment/xls",
    "application/xls",
    "application/excel",
    "application/x-excel",
    "application/x-msexcel",
}

# Politeness delay between downloads (seconds)
DOWNLOAD_DELAY = 1.5

# Request headers — mimic a real browser
HEADERS = {
    "Accept"          : "application/json, text/plain, */*",
    "Accept-Encoding" : "gzip, deflate, br",
    "Accept-Language" : "en-US,en;q=0.9",
    "Connection"      : "keep-alive",
    "Content-Type"    : "application/json",
    "Host"            : "webapi.grid-india.in",
    "Origin"          : "https://grid-india.in",
    "Referer"         : "https://grid-india.in/en/reports/daily-psp-report",
    "Sec-Fetch-Dest"  : "empty",
    "Sec-Fetch-Mode"  : "cors",
    "Sec-Fetch-Site"  : "same-site",
    "User-Agent"      : (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
}

# ── Logging helpers ────────────────────────────────────────────────────────────

def load_completed_log():
    """Return set of already-downloaded FilePaths from the log."""
    done = set()
    if LOG_FILE.exists():
        with open(LOG_FILE, newline="") as f:
            for row in csv.DictReader(f):
                if row.get("status") == "OK":
                    done.add(row["file_path"])
    return done


def append_log(file_path, year, saved_as, status, note=""):
    write_header = not LOG_FILE.exists()
    with open(LOG_FILE, "a", newline="") as f:
        w = csv.writer(f)
        if write_header:
            w.writerow(["timestamp", "year", "file_path", "saved_as", "status", "note"])
        w.writerow([
            datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            year, file_path, saved_as, status, note
        ])


# ── API helpers ────────────────────────────────────────────────────────────────

def fetch_file_list(year: str, month: str = "00") -> list:
    """
    Call the GRID-India API and return list of file records for a given year.
    month="00" means ALL months.
    """
    payload = {
        "_source"   : "GRDW",
        "_type"     : "DAILY_PSP_REPORT",
        "_fileDate" : year,
        "_month"    : month,
    }
    resp = requests.post(API_URL, json=payload, headers=HEADERS, timeout=30, verify=False)
    resp.raise_for_status()
    data = resp.json()

    if data.get("flagType") != 0:
        raise ValueError(f"API error for {year}: {data.get('retMessage')}")

    return data.get("retData", [])


def is_xls(record: dict) -> bool:
    """Return True if this record is an XLS file (not PDF)."""
    mime = (record.get("MimeType") or "").lower().strip()
    fp   = (record.get("FilePath") or "").lower()

    # Check mime type
    if any(m in mime for m in ["excel", "xls"]):
        return True
    # Fallback: check file extension
    if fp.endswith(".xls") or fp.endswith(".xlsx"):
        return True
    return False


def build_download_url(file_path: str) -> str:
    """Construct full CDN download URL from relative FilePath."""
    return CDN_BASE + file_path.lstrip("/")


def safe_filename(record: dict, year: str) -> str:
    """
    Build a clean, consistent filename:
    DD.MM.YY_NLDC_PSP_<year>.xls
    Uses Title_ field + year range suffix to avoid collisions across years.
    """
    title = record.get("Title_", "unknown").strip()
    # Sanitise — remove any chars that are bad for filenames
    title = "".join(c for c in title if c.isalnum() or c in "._- ")
    # Append year range so files from different years don't clash
    year_tag = year.replace("-", "_")
    return f"{title}__{year_tag}.xls"


# ── Main downloader ────────────────────────────────────────────────────────────

def download_file(url: str, dest: Path) -> bool:
    """Stream-download a file to dest. Returns True on success."""
    resp = requests.get(url, headers=HEADERS, timeout=60, stream=True, verify=False)
    resp.raise_for_status()
    dest.parent.mkdir(parents=True, exist_ok=True)
    with open(dest, "wb") as f:
        for chunk in resp.iter_content(chunk_size=8192):
            f.write(chunk)
    return True


def process_year(year: str, completed: set):
    print(f"\n{'═'*60}")
    print(f"  Year: {year}")
    print(f"{'═'*60}")

    # Fetch full file list for this year
    try:
        records = fetch_file_list(year)
    except Exception as e:
        print(f"  ✗ Failed to fetch file list: {e}")
        return 0, 0

    # Filter XLS only
    xls_records = [r for r in records if is_xls(r)]
    pdf_count   = len(records) - len(xls_records)

    print(f"  Total records : {len(records)}")
    print(f"  XLS files     : {len(xls_records)}")
    print(f"  PDF files     : {pdf_count}  (skipped)")

    # Output folder for this year
    year_dir = OUTPUT_DIR / year
    year_dir.mkdir(parents=True, exist_ok=True)

    ok = skip = fail = 0

    for i, rec in enumerate(xls_records, 1):
        file_path = rec.get("FilePath", "")
        title     = rec.get("Title_", "?")
        date_str  = rec.get("Field1", "?")

        # Skip if already downloaded
        if file_path in completed:
            print(f"  [{i:>4}/{len(xls_records)}] SKIP  {title} ({date_str})")
            skip += 1
            continue

        url      = build_download_url(file_path)
        fname    = safe_filename(rec, year)
        dest     = year_dir / fname

        # If dest file already exists on disk (e.g. from a previous partial run)
        if dest.exists() and dest.stat().st_size > 1000:
            print(f"  [{i:>4}/{len(xls_records)}] SKIP  {title} (file exists)")
            append_log(file_path, year, str(dest), "OK", "already on disk")
            completed.add(file_path)
            skip += 1
            continue

        try:
            download_file(url, dest)
            size_kb = dest.stat().st_size // 1024
            print(f"  [{i:>4}/{len(xls_records)}] OK    {title} ({date_str})  {size_kb} KB")
            append_log(file_path, year, str(dest), "OK")
            completed.add(file_path)
            ok += 1
        except Exception as e:
            print(f"  [{i:>4}/{len(xls_records)}] FAIL  {title} ({date_str})  → {e}")
            append_log(file_path, year, str(dest), "FAIL", str(e))
            fail += 1

        time.sleep(DOWNLOAD_DELAY)

    print(f"\n  ✓ Downloaded: {ok}  |  Skipped: {skip}  |  Failed: {fail}")
    return ok, fail


def main():
    print("=" * 60)
    print("  NLDC Daily PSP Report — XLS Downloader")
    print(f"  Years: {', '.join(YEAR_RANGES)}")
    print(f"  Output: {OUTPUT_DIR.resolve()}")
    print("=" * 60)

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    completed = load_completed_log()

    if completed:
        print(f"\n  Resuming — {len(completed)} files already downloaded previously\n")

    total_ok = total_fail = 0

    for year in YEAR_RANGES:
        ok, fail = process_year(year, completed)
        total_ok   += ok
        total_fail += fail
        # Pause between year batches
        if year != YEAR_RANGES[-1]:
            print(f"\n  Pausing 3s before next year batch...")
            time.sleep(3)

    print(f"\n{'═'*60}")
    print(f"  DONE — Total downloaded: {total_ok}  |  Failed: {total_fail}")
    print(f"  Log saved to: {LOG_FILE.resolve()}")
    print(f"  Files saved to: {OUTPUT_DIR.resolve()}/")
    print(f"{'═'*60}")

# main()

In [2]:
test_url = "https://webcdn.grid-india.in/files/grdw/2026/04/31.03.26_NLDC_PSP_819.xls"

resp1 = requests.get(test_url, headers=HEADERS, verify=False)
print(f"With our headers: {resp1.status_code}")

resp2 = requests.get(test_url, verify=False)
print(f"With no headers: {resp2.status_code}")

resp3 = requests.get(test_url, verify=False, headers={"User-Agent": "Mozilla/5.0"})
print(f"With only User-Agent: {resp3.status_code}")

resp4 = requests.get(test_url, verify=False, headers={
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Referer": "https://grid-india.in/",
    "Accept": "*/*",
})
print(f"With CDN headers: {resp4.status_code}")

With our headers: 404
With no headers: 200
With only User-Agent: 200
With CDN headers: 200


In [3]:
"""
NLDC Daily PSP Report Downloader
==================================
Downloads all XLS files from the GRID-India API for the specified year ranges.

API endpoint : POST https://webapi.grid-india.in/api/v1/file
CDN base     : https://webcdn.grid-india.in/

Usage:
    python nldc_downloader.py

All XLS files will be saved to:
    ./nldc_xls/<year_range>/<filename>.xls

A download log (nldc_download_log.csv) is written so the script can
resume safely if interrupted — already-downloaded files are skipped.
"""

import os
import time
import json
import requests
import urllib3
import csv
from pathlib import Path
from datetime import datetime

# Suppress SSL warnings — needed for grid-india.in certificate issue on Windows
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# ── Configuration ──────────────────────────────────────────────────────────────

API_URL      = "https://webapi.grid-india.in/api/v1/file"
CDN_BASE     = "https://webcdn.grid-india.in/"
OUTPUT_DIR   = Path("./nldc_xls")
LOG_FILE     = Path("./nldc_download_log.csv")

# Year ranges to download
YEAR_RANGES  = ["2025-26", "2024-25", "2023-24", "2022-23"]

# XLS mime types (exclude PDF)
XLS_MIMES    = {
    "application/vnd.ms-excel",
    "attachment/xls",
    "application/xls",
    "application/excel",
    "application/x-excel",
    "application/x-msexcel",
}

# Politeness delay between downloads (seconds)
DOWNLOAD_DELAY = 1.5

# Request headers — mimic a real browser
HEADERS = {
    "Accept"          : "application/json, text/plain, */*",
    "Accept-Encoding" : "gzip, deflate, br",
    "Accept-Language" : "en-US,en;q=0.9",
    "Connection"      : "keep-alive",
    "Content-Type"    : "application/json",
    "Host"            : "webapi.grid-india.in",
    "Origin"          : "https://grid-india.in",
    "Referer"         : "https://grid-india.in/en/reports/daily-psp-report",
    "Sec-Fetch-Dest"  : "empty",
    "Sec-Fetch-Mode"  : "cors",
    "Sec-Fetch-Site"  : "same-site",
    "User-Agent"      : (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
}

# ── Logging helpers ────────────────────────────────────────────────────────────

def load_completed_log():
    """Return set of already-downloaded FilePaths from the log."""
    done = set()
    if LOG_FILE.exists():
        with open(LOG_FILE, newline="") as f:
            for row in csv.DictReader(f):
                if row.get("status") == "OK":
                    done.add(row["file_path"])
    return done


def append_log(file_path, year, saved_as, status, note=""):
    write_header = not LOG_FILE.exists()
    with open(LOG_FILE, "a", newline="") as f:
        w = csv.writer(f)
        if write_header:
            w.writerow(["timestamp", "year", "file_path", "saved_as", "status", "note"])
        w.writerow([
            datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            year, file_path, saved_as, status, note
        ])


# ── API helpers ────────────────────────────────────────────────────────────────

def fetch_file_list(year: str, month: str = "00") -> list:
    """
    Call the GRID-India API and return list of file records for a given year.
    month="00" means ALL months.
    """
    payload = {
        "_source"   : "GRDW",
        "_type"     : "DAILY_PSP_REPORT",
        "_fileDate" : year,
        "_month"    : month,
    }
    resp = requests.post(API_URL, json=payload, headers=HEADERS, timeout=30, verify=False)
    resp.raise_for_status()
    data = resp.json()

    if data.get("flagType") != 0:
        raise ValueError(f"API error for {year}: {data.get('retMessage')}")

    return data.get("retData", [])


def is_xls(record: dict) -> bool:
    """Return True if this record is an XLS file (not PDF)."""
    mime = (record.get("MimeType") or "").lower().strip()
    fp   = (record.get("FilePath") or "").lower()

    # Check mime type
    if any(m in mime for m in ["excel", "xls"]):
        return True
    # Fallback: check file extension
    if fp.endswith(".xls") or fp.endswith(".xlsx"):
        return True
    return False


def build_download_url(file_path: str) -> str:
    """Construct full CDN download URL from relative FilePath."""
    return CDN_BASE + file_path.lstrip("/")


def safe_filename(record: dict, year: str) -> str:
    """
    Build a clean, consistent filename:
    DD.MM.YY_NLDC_PSP_<year>.xls
    Uses Title_ field + year range suffix to avoid collisions across years.
    """
    title = record.get("Title_", "unknown").strip()
    # Sanitise — remove any chars that are bad for filenames
    title = "".join(c for c in title if c.isalnum() or c in "._- ")
    # Append year range so files from different years don't clash
    year_tag = year.replace("-", "_")
    return f"{title}__{year_tag}.xls"


# ── Main downloader ────────────────────────────────────────────────────────────

def download_file(url: str, dest: Path) -> bool:
    """Stream-download a file to dest. Returns True on success.
    CDN rejects requests with Origin/Referer headers — use no headers."""
    resp = requests.get(url, timeout=60, stream=True, verify=False)
    resp.raise_for_status()
    dest.parent.mkdir(parents=True, exist_ok=True)
    with open(dest, "wb") as f:
        for chunk in resp.iter_content(chunk_size=8192):
            f.write(chunk)
    return True


def process_year(year: str, completed: set):
    print(f"\n{'═'*60}")
    print(f"  Year: {year}")
    print(f"{'═'*60}")

    # Fetch full file list for this year
    try:
        records = fetch_file_list(year)
    except Exception as e:
        print(f"  ✗ Failed to fetch file list: {e}")
        return 0, 0

    # Filter XLS only
    xls_records = [r for r in records if is_xls(r)]
    pdf_count   = len(records) - len(xls_records)

    print(f"  Total records : {len(records)}")
    print(f"  XLS files     : {len(xls_records)}")
    print(f"  PDF files     : {pdf_count}  (skipped)")

    # Output folder for this year
    year_dir = OUTPUT_DIR / year
    year_dir.mkdir(parents=True, exist_ok=True)

    ok = skip = fail = 0

    for i, rec in enumerate(xls_records, 1):
        file_path = rec.get("FilePath", "")
        title     = rec.get("Title_", "?")
        date_str  = rec.get("Field1", "?")

        # Skip if already downloaded
        if file_path in completed:
            print(f"  [{i:>4}/{len(xls_records)}] SKIP  {title} ({date_str})")
            skip += 1
            continue

        url      = build_download_url(file_path)
        fname    = safe_filename(rec, year)
        dest     = year_dir / fname

        # If dest file already exists on disk (e.g. from a previous partial run)
        if dest.exists() and dest.stat().st_size > 1000:
            print(f"  [{i:>4}/{len(xls_records)}] SKIP  {title} (file exists)")
            append_log(file_path, year, str(dest), "OK", "already on disk")
            completed.add(file_path)
            skip += 1
            continue

        try:
            download_file(url, dest)
            size_kb = dest.stat().st_size // 1024
            print(f"  [{i:>4}/{len(xls_records)}] OK    {title} ({date_str})  {size_kb} KB")
            append_log(file_path, year, str(dest), "OK")
            completed.add(file_path)
            ok += 1
        except Exception as e:
            print(f"  [{i:>4}/{len(xls_records)}] FAIL  {title} ({date_str})  → {e}")
            append_log(file_path, year, str(dest), "FAIL", str(e))
            fail += 1

        time.sleep(DOWNLOAD_DELAY)

    print(f"\n  ✓ Downloaded: {ok}  |  Skipped: {skip}  |  Failed: {fail}")
    return ok, fail


def main():
    print("=" * 60)
    print("  NLDC Daily PSP Report — XLS Downloader")
    print(f"  Years: {', '.join(YEAR_RANGES)}")
    print(f"  Output: {OUTPUT_DIR.resolve()}")
    print("=" * 60)

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    completed = load_completed_log()

    if completed:
        print(f"\n  Resuming — {len(completed)} files already downloaded previously\n")

    total_ok = total_fail = 0

    for year in YEAR_RANGES:
        ok, fail = process_year(year, completed)
        total_ok   += ok
        total_fail += fail
        # Pause between year batches
        if year != YEAR_RANGES[-1]:
            print(f"\n  Pausing 3s before next year batch...")
            time.sleep(3)

    print(f"\n{'═'*60}")
    print(f"  DONE — Total downloaded: {total_ok}  |  Failed: {total_fail}")
    print(f"  Log saved to: {LOG_FILE.resolve()}")
    print(f"  Files saved to: {OUTPUT_DIR.resolve()}/")
    print(f"{'═'*60}")

main()

  NLDC Daily PSP Report — XLS Downloader
  Years: 2025-26, 2024-25, 2023-24, 2022-23
  Output: C:\Users\Shriyans S Sahoo\DS4SG\Electricity-Demand-Forecasting\notebooks\nldc_xls

════════════════════════════════════════════════════════════
  Year: 2025-26
════════════════════════════════════════════════════════════
  Total records : 731
  XLS files     : 365
  PDF files     : 366  (skipped)
  [   1/365] OK    31.03.26_NLDC_PSP (31-03-2026)  278 KB
  [   2/365] OK    30.03.26_NLDC_PSP.xls (31-03-2026)  278 KB
  [   3/365] SKIP  30.03.26_NLDC_PSP.xls (file exists)
  [   4/365] OK    29.03.26_NLDC_PSP (30-03-2026)  278 KB
  [   5/365] OK    28.03.26_NLDC_PSP.XLS (28-03-2026)  279 KB
  [   6/365] OK    27.03.26_NLDC_PSP_H&E (27-03-2026)  277 KB
  [   7/365] OK    26.03.26_NLDC_PSP.xls (27-03-2026)  278 KB
  [   8/365] OK    25.03.26_NLDC_PSP (26-03-2026)  279 KB
  [   9/365] OK    24.03.26_NLDC_PSP (24-03-2026)  278 KB
  [  10/365] OK    23.03.26_NLDC_PSP (24-03-2026)  278 KB
  [  11/365] O

In [4]:
"""
NLDC Daily PSP Report Downloader
==================================
Downloads all XLS files from the GRID-India API for the specified year ranges.

API endpoint : POST https://webapi.grid-india.in/api/v1/file
CDN base     : https://webcdn.grid-india.in/

Usage:
    python nldc_downloader.py

All XLS files will be saved to:
    ./nldc_xls/<year_range>/<filename>.xls

A download log (nldc_download_log.csv) is written so the script can
resume safely if interrupted — already-downloaded files are skipped.
"""

import os
import time
import json
import requests
import urllib3
import csv
from pathlib import Path
from datetime import datetime

# Suppress SSL warnings — needed for grid-india.in certificate issue on Windows
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# ── Configuration ──────────────────────────────────────────────────────────────

API_URL      = "https://webapi.grid-india.in/api/v1/file"
CDN_BASE     = "https://webcdn.grid-india.in/"
OUTPUT_DIR   = Path("./nldc_xls")
LOG_FILE     = Path("./nldc_download_log.csv")

# Year ranges to download
YEAR_RANGES  = ["2025-26", "2024-25", "2023-24", "2022-23"]

# XLS mime types (exclude PDF)
XLS_MIMES    = {
    "application/vnd.ms-excel",
    "attachment/xls",
    "application/xls",
    "application/excel",
    "application/x-excel",
    "application/x-msexcel",
}

# Politeness delay between downloads (seconds)
DOWNLOAD_DELAY = 1.5

# Request headers — mimic a real browser
HEADERS = {
    "Accept"          : "application/json, text/plain, */*",
    "Accept-Encoding" : "gzip, deflate, br",
    "Accept-Language" : "en-US,en;q=0.9",
    "Connection"      : "keep-alive",
    "Content-Type"    : "application/json",
    "Host"            : "webapi.grid-india.in",
    "Origin"          : "https://grid-india.in",
    "Referer"         : "https://grid-india.in/en/reports/daily-psp-report",
    "Sec-Fetch-Dest"  : "empty",
    "Sec-Fetch-Mode"  : "cors",
    "Sec-Fetch-Site"  : "same-site",
    "User-Agent"      : (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
}

# ── Logging helpers ────────────────────────────────────────────────────────────

def load_completed_log():
    """Return set of already-downloaded FilePaths from the log."""
    done = set()
    if LOG_FILE.exists():
        with open(LOG_FILE, newline="") as f:
            for row in csv.DictReader(f):
                if row.get("status") == "OK":
                    done.add(row["file_path"])
    return done


def append_log(file_path, year, saved_as, status, note=""):
    write_header = not LOG_FILE.exists()
    with open(LOG_FILE, "a", newline="") as f:
        w = csv.writer(f)
        if write_header:
            w.writerow(["timestamp", "year", "file_path", "saved_as", "status", "note"])
        w.writerow([
            datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            year, file_path, saved_as, status, note
        ])


# ── API helpers ────────────────────────────────────────────────────────────────

def fetch_file_list(year: str, month: str = "00") -> list:
    """
    Call the GRID-India API and return list of file records for a given year.
    month="00" means ALL months.
    """
    payload = {
        "_source"   : "GRDW",
        "_type"     : "DAILY_PSP_REPORT",
        "_fileDate" : year,
        "_month"    : month,
    }
    resp = requests.post(API_URL, json=payload, headers=HEADERS, timeout=30, verify=False)
    resp.raise_for_status()
    data = resp.json()

    if data.get("flagType") != 0:
        raise ValueError(f"API error for {year}: {data.get('retMessage')}")

    return data.get("retData", [])


def is_xls(record: dict) -> bool:
    """Return True if this record is an XLS file (not PDF)."""
    mime = (record.get("MimeType") or "").lower().strip()
    fp   = (record.get("FilePath") or "").lower()

    # Check mime type
    if any(m in mime for m in ["excel", "xls"]):
        return True
    # Fallback: check file extension
    if fp.endswith(".xls") or fp.endswith(".xlsx"):
        return True
    return False


def build_download_url(file_path: str) -> str:
    """Construct full CDN download URL from relative FilePath."""
    return CDN_BASE + file_path.lstrip("/")


def safe_filename(record: dict, year: str) -> str:
    """
    Build a clean, consistent filename:
    DD.MM.YY_NLDC_PSP_<year>.xls
    Uses Title_ field + year range suffix to avoid collisions across years.
    """
    title = record.get("Title_", "unknown").strip()
    # Sanitise — remove any chars that are bad for filenames
    title = "".join(c for c in title if c.isalnum() or c in "._- ")
    # Append year range so files from different years don't clash
    year_tag = year.replace("-", "_")
    return f"{title}__{year_tag}.xls"


# ── Main downloader ────────────────────────────────────────────────────────────

def download_file(url: str, dest: Path) -> bool:
    """Stream-download a file to dest. Returns True on success.
    CDN rejects requests with Origin/Referer headers — use no headers."""
    resp = requests.get(url, timeout=60, stream=True, verify=False)
    resp.raise_for_status()
    dest.parent.mkdir(parents=True, exist_ok=True)
    with open(dest, "wb") as f:
        for chunk in resp.iter_content(chunk_size=8192):
            f.write(chunk)
    return True


def process_year(year: str, completed: set):
    print(f"\n{'═'*60}")
    print(f"  Year: {year}")
    print(f"{'═'*60}")

    # Fetch full file list for this year
    try:
        records = fetch_file_list(year)
    except Exception as e:
        print(f"  ✗ Failed to fetch file list: {e}")
        return 0, 0

    # Filter XLS only
    xls_records = [r for r in records if is_xls(r)]
    pdf_count   = len(records) - len(xls_records)

    print(f"  Total records : {len(records)}")
    print(f"  XLS files     : {len(xls_records)}")
    print(f"  PDF files     : {pdf_count}  (skipped)")

    # Output folder for this year
    year_dir = OUTPUT_DIR / year
    year_dir.mkdir(parents=True, exist_ok=True)

    ok = skip = fail = 0

    for i, rec in enumerate(xls_records, 1):
        file_path = rec.get("FilePath", "")
        title     = rec.get("Title_", "?")
        date_str  = rec.get("Field1", "?")

        # Skip if already downloaded
        if file_path in completed:
            print(f"  [{i:>4}/{len(xls_records)}] SKIP  {title} ({date_str})")
            skip += 1
            continue

        url      = build_download_url(file_path)
        fname    = safe_filename(rec, year)
        dest     = year_dir / fname

        # If dest file already exists on disk (e.g. from a previous partial run)
        if dest.exists() and dest.stat().st_size > 1000:
            print(f"  [{i:>4}/{len(xls_records)}] SKIP  {title} (file exists)")
            append_log(file_path, year, str(dest), "OK", "already on disk")
            completed.add(file_path)
            skip += 1
            continue

        try:
            download_file(url, dest)
            size_kb = dest.stat().st_size // 1024
            print(f"  [{i:>4}/{len(xls_records)}] OK    {title} ({date_str})  {size_kb} KB")
            append_log(file_path, year, str(dest), "OK")
            completed.add(file_path)
            ok += 1
        except Exception as e:
            print(f"  [{i:>4}/{len(xls_records)}] FAIL  {title} ({date_str})  → {e}")
            append_log(file_path, year, str(dest), "FAIL", str(e))
            fail += 1

        time.sleep(DOWNLOAD_DELAY)

    print(f"\n  ✓ Downloaded: {ok}  |  Skipped: {skip}  |  Failed: {fail}")
    return ok, fail


def main():
    print("=" * 60)
    print("  NLDC Daily PSP Report — XLS Downloader")
    print(f"  Years: {', '.join(YEAR_RANGES)}")
    print(f"  Output: {OUTPUT_DIR.resolve()}")
    print("=" * 60)

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    completed = load_completed_log()

    if completed:
        print(f"\n  Resuming — {len(completed)} files already downloaded previously\n")

    total_ok = total_fail = 0

    for year in YEAR_RANGES:
        ok, fail = process_year(year, completed)
        total_ok   += ok
        total_fail += fail
        # Pause between year batches
        if year != YEAR_RANGES[-1]:
            print(f"\n  Pausing 3s before next year batch...")
            time.sleep(3)

    print(f"\n{'═'*60}")
    print(f"  DONE — Total downloaded: {total_ok}  |  Failed: {total_fail}")
    print(f"  Log saved to: {LOG_FILE.resolve()}")
    print(f"  Files saved to: {OUTPUT_DIR.resolve()}/")
    print(f"{'═'*60}")

main()

  NLDC Daily PSP Report — XLS Downloader
  Years: 2025-26, 2024-25, 2023-24, 2022-23
  Output: C:\Users\Shriyans S Sahoo\DS4SG\Electricity-Demand-Forecasting\notebooks\nldc_xls

  Resuming — 895 files already downloaded previously


════════════════════════════════════════════════════════════
  Year: 2025-26
════════════════════════════════════════════════════════════
  ✗ Failed to fetch file list: ("Connection broken: ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)", ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))

  Pausing 3s before next year batch...

════════════════════════════════════════════════════════════
  Year: 2024-25
════════════════════════════════════════════════════════════
  ✗ Failed to fetch file list: HTTPSConnectionPool(host='webapi.grid-india.in', port=443): Max retries exceeded with url: /api/v1/file (Caused by NameResolutionError("<url

In [7]:
# Rerun

"""
NLDC Daily PSP Report Downloader
==================================
Downloads all XLS files from the GRID-India API for the specified year ranges.

API endpoint : POST https://webapi.grid-india.in/api/v1/file
CDN base     : https://webcdn.grid-india.in/

Usage:
    python nldc_downloader.py

All XLS files will be saved to:
    ./nldc_xls/<year_range>/<filename>.xls

A download log (nldc_download_log.csv) is written so the script can
resume safely if interrupted — already-downloaded files are skipped.
"""

import os
import time
import json
import requests
import urllib3
import csv
from pathlib import Path
from datetime import datetime

# Suppress SSL warnings — needed for grid-india.in certificate issue on Windows
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# ── Configuration ──────────────────────────────────────────────────────────────

API_URL      = "https://webapi.grid-india.in/api/v1/file"
CDN_BASE     = "https://webcdn.grid-india.in/"
OUTPUT_DIR   = Path("./nldc_xls")
LOG_FILE     = Path("./nldc_download_log.csv")

# Year ranges to download
YEAR_RANGES  = ["2025-26", "2024-25", "2023-24", "2022-23"]

# XLS mime types (exclude PDF)
XLS_MIMES    = {
    "application/vnd.ms-excel",
    "attachment/xls",
    "application/xls",
    "application/excel",
    "application/x-excel",
    "application/x-msexcel",
}

# Politeness delay between downloads (seconds)
DOWNLOAD_DELAY = 1.5

# Request headers — mimic a real browser
HEADERS = {
    "Accept"          : "application/json, text/plain, */*",
    "Accept-Encoding" : "gzip, deflate, br",
    "Accept-Language" : "en-US,en;q=0.9",
    "Connection"      : "keep-alive",
    "Content-Type"    : "application/json",
    "Host"            : "webapi.grid-india.in",
    "Origin"          : "https://grid-india.in",
    "Referer"         : "https://grid-india.in/en/reports/daily-psp-report",
    "Sec-Fetch-Dest"  : "empty",
    "Sec-Fetch-Mode"  : "cors",
    "Sec-Fetch-Site"  : "same-site",
    "User-Agent"      : (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
}

# ── Logging helpers ────────────────────────────────────────────────────────────

def load_completed_log():
    """Return set of already-downloaded FilePaths from the log."""
    done = set()
    if LOG_FILE.exists():
        with open(LOG_FILE, newline="") as f:
            for row in csv.DictReader(f):
                if row.get("status") == "OK":
                    done.add(row["file_path"])
    return done


def append_log(file_path, year, saved_as, status, note=""):
    write_header = not LOG_FILE.exists()
    with open(LOG_FILE, "a", newline="") as f:
        w = csv.writer(f)
        if write_header:
            w.writerow(["timestamp", "year", "file_path", "saved_as", "status", "note"])
        w.writerow([
            datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            year, file_path, saved_as, status, note
        ])


# ── API helpers ────────────────────────────────────────────────────────────────

def fetch_file_list(year: str, month: str = "00") -> list:
    """
    Call the GRID-India API and return list of file records for a given year.
    month="00" means ALL months.
    """
    payload = {
        "_source"   : "GRDW",
        "_type"     : "DAILY_PSP_REPORT",
        "_fileDate" : year,
        "_month"    : month,
    }
    resp = requests.post(API_URL, json=payload, headers=HEADERS, timeout=30, verify=False)
    resp.raise_for_status()
    data = resp.json()

    if data.get("flagType") != 0:
        raise ValueError(f"API error for {year}: {data.get('retMessage')}")

    return data.get("retData", [])


def is_xls(record: dict) -> bool:
    """Return True if this record is an XLS file (not PDF)."""
    mime = (record.get("MimeType") or "").lower().strip()
    fp   = (record.get("FilePath") or "").lower()

    # Check mime type
    if any(m in mime for m in ["excel", "xls"]):
        return True
    # Fallback: check file extension
    if fp.endswith(".xls") or fp.endswith(".xlsx"):
        return True
    return False


def build_download_url(file_path: str) -> str:
    """Construct full CDN download URL from relative FilePath."""
    return CDN_BASE + file_path.lstrip("/")


def safe_filename(record: dict, year: str) -> str:
    """
    Build a clean, consistent filename:
    DD.MM.YY_NLDC_PSP_<year>.xls
    Uses Title_ field + year range suffix to avoid collisions across years.
    """
    title = record.get("Title_", "unknown").strip()
    # Sanitise — remove any chars that are bad for filenames
    title = "".join(c for c in title if c.isalnum() or c in "._- ")
    # Append year range so files from different years don't clash
    year_tag = year.replace("-", "_")
    return f"{title}__{year_tag}.xls"


# ── Main downloader ────────────────────────────────────────────────────────────

def download_file(url: str, dest: Path) -> bool:
    """Stream-download a file to dest. Returns True on success.
    CDN rejects requests with Origin/Referer headers — use no headers."""
    resp = requests.get(url, timeout=60, stream=True, verify=False)
    resp.raise_for_status()
    dest.parent.mkdir(parents=True, exist_ok=True)
    with open(dest, "wb") as f:
        for chunk in resp.iter_content(chunk_size=8192):
            f.write(chunk)
    return True


def process_year(year: str, completed: set):
    print(f"\n{'═'*60}")
    print(f"  Year: {year}")
    print(f"{'═'*60}")

    # Fetch full file list for this year
    try:
        records = fetch_file_list(year)
    except Exception as e:
        print(f"  ✗ Failed to fetch file list: {e}")
        return 0, 0

    # Filter XLS only
    xls_records = [r for r in records if is_xls(r)]
    pdf_count   = len(records) - len(xls_records)

    print(f"  Total records : {len(records)}")
    print(f"  XLS files     : {len(xls_records)}")
    print(f"  PDF files     : {pdf_count}  (skipped)")

    # Output folder for this year
    year_dir = OUTPUT_DIR / year
    year_dir.mkdir(parents=True, exist_ok=True)

    ok = skip = fail = 0

    for i, rec in enumerate(xls_records, 1):
        file_path = rec.get("FilePath", "")
        title     = rec.get("Title_", "?")
        date_str  = rec.get("Field1", "?")

        # Skip if already downloaded
        if file_path in completed:
            print(f"  [{i:>4}/{len(xls_records)}] SKIP  {title} ({date_str})")
            skip += 1
            continue

        url      = build_download_url(file_path)
        fname    = safe_filename(rec, year)
        dest     = year_dir / fname

        # If dest file already exists on disk (e.g. from a previous partial run)
        if dest.exists() and dest.stat().st_size > 1000:
            print(f"  [{i:>4}/{len(xls_records)}] SKIP  {title} (file exists)")
            append_log(file_path, year, str(dest), "OK", "already on disk")
            completed.add(file_path)
            skip += 1
            continue

        try:
            download_file(url, dest)
            size_kb = dest.stat().st_size // 1024
            print(f"  [{i:>4}/{len(xls_records)}] OK    {title} ({date_str})  {size_kb} KB")
            append_log(file_path, year, str(dest), "OK")
            completed.add(file_path)
            ok += 1
        except Exception as e:
            print(f"  [{i:>4}/{len(xls_records)}] FAIL  {title} ({date_str})  → {e}")
            append_log(file_path, year, str(dest), "FAIL", str(e))
            fail += 1

        time.sleep(DOWNLOAD_DELAY)

    print(f"\n  ✓ Downloaded: {ok}  |  Skipped: {skip}  |  Failed: {fail}")
    return ok, fail


def main():
    print("=" * 60)
    print("  NLDC Daily PSP Report — XLS Downloader")
    print(f"  Years: {', '.join(YEAR_RANGES)}")
    print(f"  Output: {OUTPUT_DIR.resolve()}")
    print("=" * 60)

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    completed = load_completed_log()

    if completed:
        print(f"\n  Resuming — {len(completed)} files already downloaded previously\n")

    total_ok = total_fail = 0

    for year in YEAR_RANGES:
        ok, fail = process_year(year, completed)
        total_ok   += ok
        total_fail += fail
        # Pause between year batches
        if year != YEAR_RANGES[-1]:
            print(f"\n  Pausing 3s before next year batch...")
            time.sleep(3)

    print(f"\n{'═'*60}")
    print(f"  DONE — Total downloaded: {total_ok}  |  Failed: {total_fail}")
    print(f"  Log saved to: {LOG_FILE.resolve()}")
    print(f"  Files saved to: {OUTPUT_DIR.resolve()}/")
    print(f"{'═'*60}")

main()

  NLDC Daily PSP Report — XLS Downloader
  Years: 2025-26, 2024-25, 2023-24, 2022-23
  Output: C:\Users\Shriyans S Sahoo\DS4SG\Electricity-Demand-Forecasting\notebooks\nldc_xls

  Resuming — 979 files already downloaded previously


════════════════════════════════════════════════════════════
  Year: 2025-26
════════════════════════════════════════════════════════════
  Total records : 731
  XLS files     : 365
  PDF files     : 366  (skipped)
  [   1/365] SKIP  31.03.26_NLDC_PSP (31-03-2026)
  [   2/365] SKIP  30.03.26_NLDC_PSP.xls (31-03-2026)
  [   3/365] SKIP  30.03.26_NLDC_PSP.xls (31-03-2026)
  [   4/365] SKIP  29.03.26_NLDC_PSP (30-03-2026)
  [   5/365] SKIP  28.03.26_NLDC_PSP.XLS (28-03-2026)
  [   6/365] SKIP  27.03.26_NLDC_PSP_H&E (27-03-2026)
  [   7/365] SKIP  26.03.26_NLDC_PSP.xls (27-03-2026)
  [   8/365] SKIP  25.03.26_NLDC_PSP (26-03-2026)
  [   9/365] SKIP  24.03.26_NLDC_PSP (24-03-2026)
  [  10/365] SKIP  23.03.26_NLDC_PSP (24-03-2026)
  [  11/365] SKIP  22.03.26_NLDC

In [12]:
"""
NLDC PSP File Renamer
======================
Renames all downloaded NLDC PSP XLS files to a consistent format:
    YYYY-MM-DD_NLDC_PSP.xls

Handles all filename patterns observed:
    Pattern 1: 01.01.26_NLDC_PSP__2025_26.xls      (normal)
    Pattern 2: 01.07.25_NLDC_PSP.xls__2025_26.xls  (has .xls in middle)

Run this AFTER the downloader has finished.

Usage:
    - Set BASE_DIR to your nldc_xls folder path
    - Run in Jupyter or as a script
    - A rename log is saved to nldc_rename_log.csv
"""

import os
import re
import csv
from pathlib import Path
from datetime import datetime

# ── Configuration ──────────────────────────────────────────────────────────────

# Set this to your nldc_xls folder — update path as needed
BASE_DIR  = Path(r"C:\Users\Shriyans S Sahoo\DS4SG\Electricity-Demand-Forecasting\nldc_xls")

# Year folders to process (2022-23 already deleted)
YEAR_FOLDERS = ["2025-26", "2024-25", "2023-24"]

# Rename log
LOG_FILE  = BASE_DIR.parent / "nldc_rename_log.csv"

# ── Date parser ────────────────────────────────────────────────────────────────

def parse_date_from_filename(fname: str):
    """
    Extract date from filename.
    Expects DD.MM.YY as the first 8 characters.
    Returns (YYYY-MM-DD string, datetime object) or (None, None) on failure.
    """
    # First 8 chars should be DD.MM.YY
    raw = fname[:8]

    # Validate format with regex — must be DD.MM.YY
    if not re.match(r"^\d{2}\.\d{2}\.\d{2}$", raw):
        return None, None

    dd, mm, yy = raw.split(".")

    # Convert YY to YYYY — all files are 2023-2026
    yyyy = "20" + yy

    # Validate ranges
    try:
        dt = datetime(int(yyyy), int(mm), int(dd))
    except ValueError:
        return None, None

    return dt.strftime("%Y-%m-%d"), dt


# ── Renamer ────────────────────────────────────────────────────────────────────

def rename_folder(year_folder: str, dry_run: bool = False):
    """
    Rename all XLS files in a year folder.
    dry_run=True shows what would happen without actually renaming.
    """
    folder = BASE_DIR / year_folder
    if not folder.exists():
        print(f"  ⚠ Folder not found: {folder}")
        return []

    files = sorted(folder.glob("*.xls"))
    print(f"\n{'═'*60}")
    print(f"  Folder : {year_folder}  ({len(files)} files)")
    print(f"{'═'*60}")

    results   = []
    used_names = {}  # track duplicates within this folder

    for fpath in files:
        fname    = fpath.name
        date_str, dt = parse_date_from_filename(fname)

        if date_str is None:
            print(f"  ⚠  SKIP  (can't parse date): {fname}")
            results.append({
                "year": year_folder, "original": fname,
                "renamed_to": "", "status": "SKIP",
                "note": "Could not parse date from filename"
            })
            continue

        # Build target filename — handle duplicates
        base_target = f"{date_str}_NLDC_PSP.xls"
        if base_target not in used_names:
            target_name = base_target
            used_names[base_target] = 1
        else:
            used_names[base_target] += 1
            version = used_names[base_target]
            target_name = f"{date_str}_NLDC_PSP_v{version}.xls"

        target_path = folder / target_name

        # Skip if already correctly named
        if fname == target_name:
            print(f"  ✓  ALREADY OK  {fname}")
            results.append({
                "year": year_folder, "original": fname,
                "renamed_to": target_name, "status": "ALREADY_OK", "note": ""
            })
            continue

        # Skip if target already exists (from a previous run)
        if target_path.exists() and target_path != fpath:
            print(f"  ⚠  TARGET EXISTS  {fname}  →  {target_name}")
            results.append({
                "year": year_folder, "original": fname,
                "renamed_to": target_name, "status": "TARGET_EXISTS",
                "note": "Target file already exists — skipped"
            })
            continue

        # Do the rename
        if not dry_run:
            fpath.rename(target_path)
            print(f"  ✓  {fname}")
            print(f"     → {target_name}")
        else:
            print(f"  [DRY RUN]  {fname}")
            print(f"          → {target_name}")

        results.append({
            "year": year_folder, "original": fname,
            "renamed_to": target_name,
            "status": "DRY_RUN" if dry_run else "RENAMED",
            "note": ""
        })

    renamed = sum(1 for r in results if r["status"] == "RENAMED")
    skipped = sum(1 for r in results if r["status"] in {"SKIP", "TARGET_EXISTS", "ALREADY_OK"})
    print(f"\n  ✓ Renamed: {renamed}  |  Skipped/Already OK: {skipped}")

    return results


def save_log(all_results: list):
    with open(LOG_FILE, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["year", "original", "renamed_to", "status", "note"])
        w.writeheader()
        w.writerows(all_results)
    print(f"\n  📄 Rename log saved to: {LOG_FILE}")


# ── Main ───────────────────────────────────────────────────────────────────────

def main(dry_run=False):
    print("=" * 60)
    if dry_run:
        print("  NLDC PSP Renamer — DRY RUN (no files will be changed)")
    else:
        print("  NLDC PSP Renamer — LIVE RUN")
    print(f"  Base folder: {BASE_DIR}")
    print(f"  Year folders: {', '.join(YEAR_FOLDERS)}")
    print("=" * 60)

    all_results = []
    for year in YEAR_FOLDERS:
        results = rename_folder(year, dry_run=dry_run)
        all_results.extend(results)

    total_renamed  = sum(1 for r in all_results if r["status"] == "RENAMED")
    total_skipped  = sum(1 for r in all_results if r["status"] in {"SKIP", "TARGET_EXISTS"})
    total_ok       = sum(1 for r in all_results if r["status"] == "ALREADY_OK")

    print(f"\n{'═'*60}")
    print(f"  COMPLETE")
    print(f"  Renamed     : {total_renamed}")
    print(f"  Already OK  : {total_ok}")
    print(f"  Skipped     : {total_skipped}")
    print(f"{'═'*60}")

    if not dry_run:
        save_log(all_results)


# ── Run ────────────────────────────────────────────────────────────────────────

# STEP 1: Run dry run first to preview changes
# main(dry_run=True)

# STEP 2: When happy, run live
main(dry_run=False)

  NLDC PSP Renamer — LIVE RUN
  Base folder: C:\Users\Shriyans S Sahoo\DS4SG\Electricity-Demand-Forecasting\nldc_xls
  Year folders: 2025-26, 2024-25, 2023-24

════════════════════════════════════════════════════════════
  Folder : 2025-26  (364 files)
════════════════════════════════════════════════════════════
  ✓  01.01.26_NLDC_PSP__2025_26.xls
     → 2026-01-01_NLDC_PSP.xls
  ✓  01.02.26_NLDC_PSP__2025_26.xls
     → 2026-02-01_NLDC_PSP.xls
  ✓  01.03.26_NLDC_PSP__2025_26.xls
     → 2026-03-01_NLDC_PSP.xls
  ✓  01.04.25_NLDC_PSP__2025_26.xls
     → 2025-04-01_NLDC_PSP.xls
  ✓  01.05.25_NLDC_PSP__2025_26.xls
     → 2025-05-01_NLDC_PSP.xls
  ✓  01.06.25_NLDC_PSP__2025_26.xls
     → 2025-06-01_NLDC_PSP.xls
  ✓  01.07.25_NLDC_PSP.xls__2025_26.xls
     → 2025-07-01_NLDC_PSP.xls
  ✓  01.08.25_NLDC_PSP.xls__2025_26.xls
     → 2025-08-01_NLDC_PSP.xls
  ✓  01.09.25_NLDC_PSP__2025_26.xls
     → 2025-09-01_NLDC_PSP.xls
  ✓  01.10.25_NLDC_PSP__2025_26.xls
     → 2025-10-01_NLDC_PSP.xls
  ✓  01.